In [3]:

# print(f"Dataset extracted to {extract_path}")
import zipfile
import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# Install gdown to download from Google Drive
!pip install -q gdown

file_id = "12iO6rIrPRmrJbM00jErCtp5J5ICWbp4i"
zip_path = "dataset_reconstructions.zip"

# Download the file using gdown
import gdown
gdown.download(id=file_id, output=zip_path, quiet=False)

# Extract the zip file
extract_path = "/content"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"✅ Dataset extracted to {extract_path}")

Downloading...
From (original): https://drive.google.com/uc?id=12iO6rIrPRmrJbM00jErCtp5J5ICWbp4i
From (redirected): https://drive.google.com/uc?id=12iO6rIrPRmrJbM00jErCtp5J5ICWbp4i&confirm=t&uuid=ffeb7ef1-ecea-4177-96b3-b26244e91ea8
To: /content/dataset_reconstructions.zip
100%|██████████| 111M/111M [00:00<00:00, 195MB/s]


✅ Dataset extracted to /content


In [4]:


def load_reconstruction_dataset(root_dir):
    """Load the organized reconstruction dataset"""
    samples = []

    # Find all sample directories
    sample_dirs = sorted([d for d in os.listdir(root_dir) if d.startswith("sample_")])

    for sample_dir in sample_dirs:
        sample_path = os.path.join(root_dir, sample_dir)

        # Load original slices
        orig_dir = os.path.join(sample_path, "original")
        orig_slices = sorted([f for f in os.listdir(orig_dir) if f.endswith(".npy")])
        original = np.stack([np.load(os.path.join(orig_dir, f)) for f in orig_slices])

        # Load reconstructed slices
        recon_dir = os.path.join(sample_path, "reconstructed")
        recon_slices = sorted([f for f in os.listdir(recon_dir) if f.endswith(".npy")])
        reconstructed = np.stack([np.load(os.path.join(recon_dir, f)) for f in recon_slices])

        samples.append({
            'original': original,
            'reconstructed': reconstructed,
            'name': sample_dir
        })

    return samples

def interactive_visualization(samples):
    """Create interactive visualization with ipywidgets sliders"""
    def display_images(sample_idx, slice_idx):
        """Display the original and reconstructed images for the given sample and slice"""
        sample = samples[sample_idx]

        # Original image
        orig_img = np.squeeze(sample['original'][slice_idx])  # Remove singleton dimension
        recon_img = np.squeeze(sample['reconstructed'][slice_idx])  # Remove singleton dimension

        # Plot the images
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
        ax1.imshow(orig_img, cmap='viridis')
        ax1.set_title(f"Original\n{sample['name']}\nSlice {slice_idx}")
        ax2.imshow(recon_img, cmap='viridis')
        ax2.set_title(f"Reconstructed\n{sample['name']}\nSlice {slice_idx}")
        plt.show()

    # Create interactive sliders
    interact(
        display_images,
        sample_idx=IntSlider(min=0, max=len(samples)-1, step=1, value=0, description='Sample'),
        slice_idx=IntSlider(min=0, max=samples[0]['original'].shape[0]-1, step=1, value=0, description='Slice')
    )

# Main function for Colab
def main(data_dir):
    # Load the data
    samples = load_reconstruction_dataset(data_dir)
    print(f"Loaded {len(samples)} samples")

    # Start interactive visualization
    interactive_visualization(samples)

In [5]:
# Path to the extracted dataset
data_dir = '/content/dataset_reconstructions'

# Run the visualization
main(data_dir)

Loaded 20 samples


interactive(children=(IntSlider(value=0, description='Sample', max=19), IntSlider(value=0, description='Slice'…